In [ ]:
import re

import pandas as pd

# 读取Excel文件
df_glossary = pd.read_excel("glossary_all.xlsx")

# 只保留需要的两列
df_glossary = df_glossary[["zh-CN", "ko"]]

# 删除任何包含NaN的行
df_glossary = df_glossary.dropna(how='any')

# 按照中文列去重
df_glossary = df_glossary.drop_duplicates(subset="zh-CN", keep="first")

# 确保'zh-CN'和'ko'列中的所有元素都是字符串
df_glossary['zh-CN'] = df_glossary['zh-CN'].astype(str)
df_glossary['ko'] = df_glossary['ko'].astype(str)

# 删除包含数字的行
mask = df_glossary.applymap(lambda x: bool(re.search(r'\d', x)))
df_glossary = df_glossary[~mask.any(axis=1)]

#按照长度降序排布
df_glossary["zh-CN_length"] = df_glossary["zh-CN"].str.len()
df_glossary = df_glossary.sort_values("zh-CN_length", ascending=False)
df_glossary = df_glossary.drop('zh-CN_length', axis=1)
df_glossary

In [ ]:
df_glossary.set_index("zh-CN", inplace=True)
glossary_dict = df_glossary["ko"].to_dict()
glossary_dict

Load the tokenizer:

In [ ]:
from transformers import AutoTokenizer

model_name = "autodl-tmp/facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, tgt_lang=None)

Add special tokens to tokenizer (for glossary translation):

In [ ]:
additional_special_tokens = tokenizer.additional_special_tokens
additional_special_tokens.extend(["<start>", "<middle>", "<end>"])
additional_special_tokens.extend(["<code_id={0}>".format(id) for id in range(10)])
additional_special_tokens

In [ ]:
special_tokens_dict = {'additional_special_tokens': additional_special_tokens}
tokenizer.add_special_tokens(special_tokens_dict)

Prepare your training and validation data:

In [ ]:
rename_dict = {"ja": "jpn_Jpan",
               "en": "eng_Latn",
               "zh-CN": "zho_Hans",
               "th": "tha_Thai",
               "ru": "rus_Cyrl",
               "zh-TW": "zho_Hant",
               "id": "ind_Latn",
               "ko": "kor_Hang",
               "pt": "por_Latn", }

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_excel("all_files_merged.xlsx")
df.drop("source", axis=1, inplace=True)  # delete the "source" column
df.replace(np.nan, None, inplace=True)  # replace np.nan with None
df.rename(columns=rename_dict, inplace=True)  # rename the columns
df

In [ ]:
language_pair = ["zho_Hans", "kor_Hang"]
df = df[language_pair]
df = df.dropna(how='any')  # clear all rows include nan
df = df.drop_duplicates()  # delete exactly the same rows
df

find the max length of the encodings

In [ ]:
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

train_encodings_list = []
valid_encodings_list = []

language_x, language_y = language_pair[0], language_pair[1]
texts_language_x = df[language_x].to_list()
texts_language_y = df[language_y].to_list()
# pre-processing (replace glossaries)
for index in tqdm(range(len(texts_language_x))):
    text_source, text_target = texts_language_x[index], texts_language_y[index]
    # code special tags
    suffix_tags = list(
        set(re.findall("|".join(["\[\/.*?\]", "\<\/.*?\>"]),
                       text_source + text_target)))  # e.g. ['[/color]', '[/size]']
    suffix_tag_types = [re.search("\w+", tag).group() for tag in suffix_tags]  # e.g.['size', 'color']
    regex = "|".join(
        [r"\[{0}=.*?\]".format(tag_type) for tag_type in suffix_tag_types] +
        [r"\<{0}=.*?\>".format(tag_type) for tag_type in suffix_tag_types])  # e.g.'\[size=.*?\]|\[color=.*?\]'
    prefix_tags = list(
        set(re.findall(regex, text_source + text_target) if regex else []))  # e.g. ['[color=#F7C358]', '[size=40]']
    tags_all = suffix_tags + prefix_tags  # e.g. ['[/color]', '[/size]', '[color=#F7C358]', '[size=40]']
    for id, tag in enumerate(tags_all):
        text_source = text_source.replace(tag, "<code_id={0}>".format(id))
        text_target = text_target.replace(tag, "<code_id={0}>".format(id))

    # glossary special tags
    for glossary_source, glossary_target in glossary_dict.items():
        if glossary_source in text_source and glossary_target in text_target and bool(
                re.search(r"\<start\>.*?{0}.*?\<middle\>".format(glossary_source), text_source)) is False:
            text_source = text_source.replace(glossary_source,
                                              "<start>{0}<middle>{1}<end>".format(glossary_source, glossary_target))
            text_source_temp = text_source.replace(glossary_source, "")
            text_target = text_target.replace(glossary_target, "<start>{0}<end>".format(glossary_target))
            text_target_temp = text_target.replace(glossary_source, "")

    texts_language_x[index] = text_source
    texts_language_y[index] = text_target

x_train, x_valid, y_train, y_valid = train_test_split(texts_language_x, texts_language_y, test_size=0.2,
                                                      random_state=42)

pre-processed data(all) saving:

In [ ]:
df_preprocessed_all = pd.DataFrame({"zh-CN": texts_language_x, "ko": texts_language_y})
df_preprocessed_all.to_excel("all_files_merged_zh-CN_ko_all_tagged.xlsx", index=False)
df_preprocessed_all

pre-processed data(valid) saving:

In [ ]:
df_preprocessed_valid = pd.DataFrame({"zh-CN": x_valid, "ko": y_valid})
df_preprocessed_valid.to_excel("all_files_merged_zh-CN_ko_valid_tagged.xlsx", index=False)
df_preprocessed_valid

In [ ]:
# add basic chars not in the vocabulary of the tokenizer
chars = list(set(''.join(x_train + y_train)))
chars_not_in_vocab = [char for char in chars if 3 in tokenizer(char).input_ids]  # 3 is the value of unknown words
tokenizer.add_tokens(chars_not_in_vocab)

the method of add vocab with sentencepiece:

In [ ]:
# import sentencepiece as spm
# 
# # use SentencePiece to get vocabulary and add into the tokenizer
# texts_filename = "{0}_{1}_texts.txt".format(language_x, language_y)
# with open(texts_filename, "w", encoding="utf-8") as file:
#     file.writelines([text + "\n" for text in x_train + y_train])
# 
# model_prefix = '{0}.{1}.sentencepiece.bpe'.format(language_x, language_y)
# vocab_size = 4200
# spm.SentencePieceTrainer.train(input=texts_filename, model_prefix=model_prefix, vocab_size=vocab_size)
# 
# with open("{0}.vocab".format(model_prefix), "r", encoding="utf-8") as file:
#     tokens = [token.strip().split("\t")[0] for token in file.readlines()]
# 
# tokenizer.add_tokens(tokens)

In [ ]:
# Note: we're now creating separate encodings for the inputs and outputs.
# truncation: truncate the sequence to a shorter length, because sometimes a sequence may be too long for a model to handle
# padding: Padding is a strategy for ensuring tensors are rectangular by adding a special padding token to shorter sentences.
#     True or 'longest': Pad to the longest sequence in the batch (or no padding if only a single sequence if provided).
#     'max_length': Pad to a maximum length specified with the argument max_length or to the maximum acceptable input length for the model if that argument is not provided.
#     False or 'do_not_pad' (default): No padding (i.e., can output a batch with sequences of different lengths).
# return_tensors: If set 'pt', will return tensors instead of list of python integers. Acceptable values are PyTorch torch.Tensor objects.
# max_length (int, optional): Controls the maximum length to use by one of the truncation/padding parameters.
# 注意：一定要注意这个max_length的使用，当不同的批次要堆叠在一起时，不可以设置为True，而是应该设置为‘max_length'，这样它才能被填充/截断到同一个长度
tokenizer.src_lang = language_x
tokenizer.tgt_lang = language_y
print("tokenizing - source:{}, target:{}".format(language_x, language_y))
train_encodings = tokenizer(x_train, text_target=y_train, truncation=True, padding="max_length",
                            return_tensors="pt")
valid_encodings = tokenizer(x_valid, text_target=y_valid, truncation=True, padding="max_length",
                            return_tensors="pt")

train_encodings_list.append(train_encodings)
valid_encodings_list.append(valid_encodings)

len(train_encodings_list)

Convert your encodings into torch Datasets object:

In [ ]:
import torch


class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, data_encoded_list: list):
        self.input_ids = []
        self.attention_mask = []
        self.labels = []
        for data_encoded in data_encoded_list:
            self.input_ids.extend(data_encoded.data["input_ids"])
            self.attention_mask.extend(data_encoded.data["attention_mask"])
            self.labels.extend(data_encoded.data["labels"])

    def __getitem__(self, index):
        item = {"input_ids": self.input_ids[index],
                "attention_mask": self.attention_mask[index],
                "labels": self.labels[index]}
        return item

    def __len__(self):
        return len(self.input_ids)


train_dataset = TranslationDataset(train_encodings_list)
valid_dataset = TranslationDataset(valid_encodings_list)

Load the pretrained model:

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")

In [ ]:
# 调整embedding层的大小
model.resize_token_embeddings(len(tokenizer))

Define your training arguments and train the model:

In [ ]:
custom_model_name = "autodl-tmp/nllb-200-distilled-600M/zh2ko"

In [ ]:
from transformers import Trainer, TrainingArguments, IntervalStrategy

# fp16：半精度运算，启用后提高一倍以上运算速度，不影响loss
# gradient_accumulation_steps：steps越大，速度越快，loss越高
# gradient_checkpointing：启用后，降低30%左右速度，节省显存2/3
# per_device_train_batch_size：size越大，GPU占用率越大，速度越快，loss越高，几乎成正比
training_args = TrainingArguments(custom_model_name,
                                  num_train_epochs=10,
                                  per_device_eval_batch_size=1,
                                  per_device_train_batch_size=1,
                                  gradient_accumulation_steps=1,
                                  gradient_checkpointing=True,
                                  fp16=True,
                                  warmup_ratio=0.1,
                                  evaluation_strategy=IntervalStrategy.STEPS,
                                  eval_steps=10000,
                                  logging_strategy=IntervalStrategy.STEPS,
                                  logging_steps=10000,
                                  save_strategy=IntervalStrategy.STEPS,
                                  save_steps=10000,
                                  save_total_limit=1,
                                  load_best_model_at_end=True,
                                  )
from torch.utils import checkpoint  #未知的bug：不会自动加载这个包

In [ ]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,  # the instantiated 🤗 Transformers model to be trained
    args=training_args,  # training arguments, defined above
    train_dataset=train_dataset,  # training dataset
    eval_dataset=valid_dataset,  # valid dataset
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train(resume_from_checkpoint=False)

Save your fine-tuned model and tokenizer:

In [ ]:
trainer.save_model(custom_model_name)
tokenizer.save_pretrained(custom_model_name)